# Procedural Frozen Lake with explicit Q* metadata

This notebook shows one custom environment and one environment tool together: `Procedural-FrozenLake-v1` plus an explicit `q_star_source`. The wrapper stack adds exact Q-values to `result[i]["q_star"]` each step, so a rollout can choose greedy actions without a separate policy network.

## Imports

Discrete actions are passed as plain dictionaries with an `action` field (one integer tensor per env).

In [ ]:
import numpy as np
import torch

from mouse_envs import EnvConfig, make_vector_env

## Environment with explicit Q* source

We pass `step_penalty=-0.01`. The env solves with `gamma=1.0`, so without a per-step cost every hole-avoiding action shares the same optimal value `Q* = 1.0` — Q* would tell the agent how to avoid holes but not how to make progress, and greedy tie-breaking can wander until the step limit. A small step penalty makes value iteration prefer shorter paths, so greedy `argmax` becomes a true shortest-path expert that reaches the goal every episode.

In [ ]:
cfg = EnvConfig(
    id="Procedural-FrozenLake-v1",
    seed=7,
    num_envs=1,
    max_episode_steps=200,
    q_star_source={"provider": "metadata_q_star"},
    kwargs={"step_penalty": -0.01},
)
env = make_vector_env(cfg)

## Inspect the reset frame

The first `step` performs the internal reset and already includes `q_star` for the start state.

In [ ]:
outputs, metrics = env.step(env.sample_random_inputs())

print(f"observation: {outputs[0]['observation'].tolist()}")
print(f"q_star:      {outputs[0]['q_star']}")

## Expert rollout

Take `argmax` over Q* to choose greedy actions, then feed those actions back into the same step-only API.

In [ ]:
episodes = 0

for step in range(200):
    inputs = [
        {
            "action": torch.tensor(
                int(np.argmax(outputs[i]["q_star"])),
                dtype=torch.int64,
            )
        }
        for i in range(env.num_envs)
    ]
    outputs, metrics = env.step(inputs)

    r = outputs[0]
    if r["done"].item() != 0:
        episodes += 1
        done_code = r["done"].item()
        outcome = "terminated" if done_code == 1 else "truncated"
        print(
            f"step={step:3d}  episode={episodes}  outcome={outcome}  "
            f"reward={r['reward'].item():.1f}"
        )

## With vs without the expert

To see what `q_star` buys you, run many episodes two ways and compare episode success rate (FrozenLake gives `reward = 1` only on reaching the goal): **with the expert** (greedy `argmax` over `q_star`) vs **without** (random actions). The greedy-Q* policy needs no trained network — it reads the exact Q-values the wrapper attaches each step.

In [ ]:
import matplotlib.pyplot as plt


def evaluate(use_expert, *, steps=800, seed=7, num_envs=64):
    cfg = EnvConfig(
        id="Procedural-FrozenLake-v1",
        seed=seed,
        num_envs=num_envs,
        max_episode_steps=200,
        q_star_source={"provider": "metadata_q_star"},
        kwargs={"step_penalty": -0.01},
    )
    env = make_vector_env(cfg)
    outputs, metrics = env.step(env.sample_random_inputs())
    episode_rewards = []
    for _ in range(steps):
        if use_expert:
            inputs = [
                {
                    "action": torch.tensor(
                        int(np.argmax(outputs[i]["q_star"])),
                        dtype=torch.int64,
                    )
                }
                for i in range(env.num_envs)
            ]
        else:
            inputs = env.sample_random_inputs()
        outputs, metrics = env.step(inputs)
        for m in metrics:
            episode_rewards.extend(m["episode_cum_reward"])
    env.close()
    return np.array(episode_rewards)


expert_eps = evaluate(use_expert=True)
random_eps = evaluate(use_expert=False)

success = [float((expert_eps > 0).mean()), float((random_eps > 0).mean())]
labels = [f"with expert\n(n={len(expert_eps)})", f"without expert\n(n={len(random_eps)})"]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, success, color=["#2a9d8f", "#e76f51"])
ax.set_ylim(0, 1.1)
ax.set_ylabel("episode success rate (reward > 0)")
ax.set_title("Procedural FrozenLake: greedy Q* vs random")
for bar, rate in zip(bars, success):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 0.02, f"{rate:.0%}", ha="center")
fig.tight_layout()
plt.show()

## Cleanup

In [ ]:
env.close()